In [1]:
"""
HR Employee Data Validation

Project Overview:
This project demonstrates data quality checks and cleaning for an Employee dataset.
It includes validation of joining dates, salary ranges, duplicate Employee IDs, 
and standardization of department names.

Tech Stack: Python, Pandas, SQL (for extension)
"""

# Import Libraries
import pandas as pd
import numpy as np
from datetime import datetime

# Simulate Employee Dataset

np.random.seed(42)
num_records = 50

departments = ["HR", "Finance", "IT", "Sales", "Operations", "hr", "finance", "it", "sales", "ops"]
employee_ids = np.random.randint(1000, 1050, num_records)  # may include duplicates
join_dates = pd.date_range(start="2015-01-01", periods=num_records, freq="200D").to_series().astype(str)

# Introduce some invalid/future dates
join_dates.iloc[5] = "2035-01-01"   # future date
join_dates.iloc[10] = "invalid_date"  # invalid

salaries = np.random.randint(20000, 120000, num_records).astype(float)
salaries[3] = -5000   # invalid negative salary
salaries[15] = 10000000  # unrealistic salary

data = {
    "EmployeeID": employee_ids,
    "Name": [f"Employee_{i}" for i in range(num_records)],
    "Department": np.random.choice(departments, num_records),
    "JoiningDate": join_dates.values,
    "Salary": salaries
}

df = pd.DataFrame(data)
print("Raw Employee Data Sample:\n", df.head(), "\n")

# Data Validation & Cleaning

# 1. Validate Date Format & Future Dates
def validate_date(date_str):
    try:
        dt = pd.to_datetime(date_str, errors="raise")
        if dt > datetime.today():
            return np.nan   # future date set to NaN
        return dt
    except:
        return np.nan

df["JoiningDate_Clean"] = df["JoiningDate"].apply(validate_date)

# 2. Validate Salary Range (keep between 20k–200k)
df["Salary_Clean"] = df["Salary"].apply(lambda x: x if 20000 <= x <= 200000 else np.nan)

# 3. Remove Duplicate EmployeeIDs
duplicates_count = df.duplicated(subset=["EmployeeID"]).sum()
df = df.drop_duplicates(subset=["EmployeeID"])

# 4. Standardize Department Names
df["Department_Clean"] = df["Department"].str.title().replace({
    "Ops": "Operations"
})

# Summary Report
report = {
    "Total Records After Cleaning": len(df),
    "Invalid/Future Dates Fixed": df["JoiningDate_Clean"].isna().sum(),
    "Invalid Salaries Fixed": df["Salary_Clean"].isna().sum(),
    "Duplicate EmployeeIDs Removed": duplicates_count
}

print("\nData Cleaning Summary Report:")
for k, v in report.items():
    print(f"{k}: {v}")

# Save Cleaned Dataset

df.to_csv("cleaned_employee_data.csv", index=False)
print("\n Cleaned dataset saved as 'cleaned_employee_data.csv'")


Raw Employee Data Sample:
    EmployeeID        Name Department JoiningDate    Salary
0        1038  Employee_0         it  2015-01-01   88148.0
1        1028  Employee_1         HR  2015-07-20   43483.0
2        1014  Employee_2         it  2016-02-05   68555.0
3        1042  Employee_3         it  2016-08-23   -5000.0
4        1007  Employee_4         IT  2017-03-11  100077.0 


Data Cleaning Summary Report:
Total Records After Cleaning: 35
Invalid/Future Dates Fixed: 20
Invalid Salaries Fixed: 2
Duplicate EmployeeIDs Removed: 15

 Cleaned dataset saved as 'cleaned_employee_data.csv'
